In [ ]:
from benchmark_pipeline.path import DATA
from benchmark_pipeline.config import (
    FEATURES, SEED, MAX_QUERY_NODES,
    CITY_BBOXES,
    CITY_ECRS,
    MAX_NEW_QUERIES,
    RATIO_FEATURES,
    NB_QUERY_TEMPLATE,
    NB_QUERY_FEATURE,
    RATIO_WITH_FEATURES,
    IS_PARAPHRASE,
    NB_SEED_QUESTIONS,
    TEMPLATE_MULTIHOP,
)
from benchmark_pipeline.generator.features import *

from benchmark_pipeline.loader.osm_loaders import load_tables
from benchmark_pipeline.loader.data_cleaning.name_matching import name_matching_cleaning
from benchmark_pipeline.generator.template_question.composite.generator_qcomposite import make_question_composite
from benchmark_pipeline.generator.template_question.composite.format_samples import format_composit_samples
from benchmark_pipeline.generator.template_question.register import register, BenchmarkConfig
from benchmark_pipeline.generator.template_question.multihop.generator_multihop import make_multihop_dataset

from benchmark_pipeline.generator.targets.mask_anchors import add_anchors_mask
from benchmark_pipeline.generator.targets.ellypses import add_ellypse
from benchmark_pipeline.generator.targets.anchors import add_anchors
from benchmark_pipeline.dataviz_tools.question_viz import montrer_questions

In [ ]:
for city_name, bbox in CITY_BBOXES.items():
    ecrs = CITY_ECRS[city_name]
    df_osm, df_street, df_area = load_tables(bbox, ecrs)
    df_all, (df_osm_clean, df_street_clean, df_area_clean) = name_matching_cleaning(df_osm, df_street, df_area)

    df_question = make_question_composite(
        df_osm_clean, 
        df_area_clean,
        df_street_clean, 
        nb_q_by_temp=NB_QUERY_TEMPLATE, 
        nb_q_by_feat=NB_QUERY_FEATURE, 
        ratio_features=RATIO_FEATURES,
        part_avec_features=RATIO_WITH_FEATURES,
        feature_cols=FEATURES,
        seed=SEED,
        is_paraphrase=IS_PARAPHRASE,
        
    )
    conf = BenchmarkConfig(
            ratio_test = 0.1,
            ratio_features = None, #{1: 0.4, 2: 0.4, 3: 0.2}, 
            ratio_category= None,
            rows_target = [],#["mask_token_anchors", "ellypse", "anchors_type_target"],
            nb_hard = 20,
            nb_unmatches = 1,
            seed = SEED,
        )
    register(DATA / city_name, df_question, df_osm_clean, df_street_clean, df_area_clean, df_all, conf, format_composit_samples, overwritte=True)
    make_multihop_dataset(    
        NB_SEED_QUESTIONS,
        city_name,
        TEMPLATE_MULTIHOP,
        df_question,
        df_osm_clean, 
        df_street_clean, 
        df_area_clean,
        )

In [ ]:
"""    #data= make_paraphrase(df_question_test, nb_generated=10)

    is_gseek_training = False
    if is_gseek_training:
        #sentence_model = SentenceEncoder()
        #df_question = add_anchors_mask(df_question, sentence_model)
        df_question = add_ellypse(df_question)
        df_question = add_anchors(df_question)
    #df_question = df_question.rename(columns={"pois": "results_poi_id"})
    # """

In [ ]:
#montrer_questions(df_question, df_osm, par_type=True)